In [1]:
# run cell to install prokbert lib (if necessary)
# !pip install git+https://github.com/nbrg-ppcu/prokbert.git@development --quiet

In [ ]:
import torch
import datasets
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, balanced_accuracy_score
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

from prokbert.tokenizer import LCATokenizer
from prokbert.sequence_dataset import SequenceDataset
from prokbert.data_collator import SegmentDataCollator
from prokbert.dataset import RandomSegmentDataset, ContiguousSegmentDataset


In [2]:
base_dir = '/home/datasets/experiments/experiment-bioinfo/'

In [22]:
train_ds = datasets.load_dataset("neuralbioinfo/PhaStyle-SequenceDB", split="BACPHLIP_TRAINING")
test_ds  = datasets.load_dataset("neuralbioinfo/PhaStyle-SequenceDB", split="BACPHLIP_VALIDATION")

In [24]:
print(f"train examples: {len(train_ds)}, columns: {train_ds.column_names}")
print(f"test examples: {len(test_ds)}, columns: {test_ds.column_names}")

train examples: 1798, columns: ['sequence_id', 'dataset', 'class_label', 'y', 'source_fasta_id', 'orientation', 'test_fastaid', 'L_seq', 'source_description', 'Genome type', 'Family', 'subds_seqid', 'sequence', '__index_level_0__']
test examples: 316, columns: ['sequence_id', 'dataset', 'class_label', 'y', 'source_fasta_id', 'orientation', 'test_fastaid', 'L_seq', 'source_description', 'Genome type', 'Family', 'subds_seqid', 'sequence', '__index_level_0__']


In [25]:
# remove cols
cols = ["dataset", "class_label", "test_fastaid", "L_seq", 'source_description', 'Genome type', 'Family', 'subds_seqid', "__index_level_0__"]

train_ds = train_ds.remove_columns(cols)
test_ds = test_ds.remove_columns(cols)

In [26]:
# rename y to label
train_ds = train_ds.rename_column("y", "label")
train_ds = train_ds.rename_column("source_fasta_id", "contig_id")
test_ds = test_ds.rename_column("y", "label")
test_ds = test_ds.rename_column("source_fasta_id", "contig_id")

In [47]:
# train_ds[0]

In [28]:
train_save_dir = f"{base_dir}/example-train-dataset/"
test_save_dir = f"{base_dir}/example-test-dataset/"

In [29]:
# train_ds.save_to_disk(train_save_dir)
# test_ds.save_to_disk(test_save_dir)

In [30]:
train_seq_ds = SequenceDataset()

save_dir = f"{base_dir}/example-train-sequence-dataset"

train_seq_ds.create_dataset([train_save_dir], save_dir=save_dir)

print("Total length:", len(train_seq_ds.sequence))

Total length: 116680168


In [31]:
pd.DataFrame(train_seq_ds.metadata).head()

,genome_id,contig_id,sequence_id,coordinate,orientation,description,label
0,None,PHAA0000323,0,"(0, 53124)",forward,None,0
1,None,PHAA0000323,1,"(53124, 106248)",reverse,None,0
2,None,PHAA0000761,2,"(106248, 177813)",forward,None,1
3,None,PHAA0000761,3,"(177813, 249378)",reverse,None,1
4,None,PHAA0000840,4,"(249378, 302774)",forward,None,0


In [32]:
test_seq_ds = SequenceDataset()

save_dir = f"{base_dir}/example-test-sequence-dataset"

test_seq_ds.create_dataset([test_save_dir], save_dir=save_dir)

print("Total length:", len(test_seq_ds.sequence))

Total length: 24854660


In [33]:
pd.DataFrame(test_seq_ds.metadata).head()

,genome_id,contig_id,sequence_id,coordinate,orientation,description,label
0,None,PHAA0000415,0,"(0, 168903)",forward,None,1
1,None,PHAA0000415,1,"(168903, 337806)",reverse,None,1
2,None,PHAA0000088,2,"(337806, 398748)",forward,None,0
3,None,PHAA0000088,3,"(398748, 459690)",reverse,None,0
4,None,PHAA0000253,4,"(459690, 521360)",forward,None,0


In [34]:
# quick check
for i, meta in enumerate(train_seq_ds.metadata):
    start, end = meta["coordinate"]
    print(f"{i:3}: {meta['contig_id']:12} [{start:3}, {end:3})  {train_seq_ds.sequence[start:end]}")
    if i >= 10: break

  0: PHAA0000323  [  0, 53124)  ...
  1: PHAA0000323  [53124, 106248)  ...
  2: PHAA0000761  [106248, 177813)  ...
  3: PHAA0000761  [177813, 249378)  ...
  4: PHAA0000840  [249378, 302774)  ...
  5: PHAA0000840  [302774, 356170)  ...
  6: PHAA0000271  [356170, 427735)  ...
  7: PHAA0000271  [427735, 499300)  ...
  8: PHAA0000060  [499300, 551830)  ...
  9: PHAA0000060  [551830, 604360)  ...
 10: PHAA0000180  [604360, 673210)  ...


In [35]:
# parameters
min_segment_length = 256
max_segment_length = 512

kmer = 6
shift = 2
add_special_tokens = False
return_metadata = False

batch_size = 16

In [36]:
train_dataset = RandomSegmentDataset(
    sequence_dataset=train_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    return_sequence=True,
)
test_dataset = ContiguousSegmentDataset(
    sequence_dataset=test_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    return_sequence=True
)

In [37]:
for i, segment in enumerate(iter(train_dataset)):
    if i >= 5: break
    print(segment)

{'segment_id': 0, 'contig_id': 'PHAA0000224', 'genome_id': None, 'sequence_id': 1039, 'absolute_coordinate': (62094054, 62094566), 'relative_coordinate': (20387, 20899), 'orientation': 'reverse', 'label': 1, 'sequence': 'CCTATGCTATGACTCAGTTACCCAAACGCCTTGGAGCGAATGGAGGATTTACTCTTGTTCTTGGCTGCGCTGTCATCCAGCTTTACCTTGAGACTGCTCTTACCGCTGCTCGGTGTGGTTGGCACCTCAGAGGAAGACACACTGGTGTCCTCGTCATCATCGCCTCCCCATACCACAGACTTCGGAGGTTCAGTCAAAGGTGCTGGTTCAGCAGCGCGAATCTGGTTAGTGTCCATCTTAGGGACTTTCACCTTCGGTTTCCAGCACATATCGTTACTCCTGTCTCAGTTGCTCTTTACGCACCTCAATCTCATCAAGTGTTCGAGAGGCAAGGTACAGACCATGCATCACACCCGCAATGAAAGACTCAGAGTAACCAGCAGCGCGGAGCGCACGGTACTCCCCGGACTCCATCACATAAGCCTGATTGAAGCGCACCTGTAGATACTCTTTAGCTGCACGCGGCATACTAGGAATATCATCGGGATGAGTTAAAACATGATTAATTGGTT'}
{'segment_id': 1, 'contig_id': 'PHAA0000968', 'genome_id': None, 'sequence_id': 844, 'absolute_coordinate': (50124918, 50125430), 'relative_coordinate': (13030, 13542), 'orientation': 'forward', 'label': 0, 'sequence': 'GTGAATCTATTACAGACTTTACAAGAAAAACAAGCAAGGTCAAT

In [38]:
tokenizer = LCATokenizer(kmer = kmer, shift = shift)
collator = SegmentDataCollator(
    tokenizer=tokenizer, add_special_tokens=add_special_tokens, return_metadata=return_metadata
)
loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collator)

In [39]:
# quick check of batch shapes
batch = next(iter(loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 254])
torch.Size([16, 254])
torch.Size([16])


In [40]:
model_name = "neuralbioinfo/prokbert-mini-long"
num_labels = len(train_ds.unique("label")) # 2

In [41]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels, trust_remote_code=True
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):   # some models also return hidden states
        logits = logits[0]
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
        "mcc": matthews_corrcoef(labels, preds),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=f"{base_dir}/prokbert-mini-long-phastyle-test",
    max_steps=2_000,  # required: the random dataset is infinite
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 4,
    learning_rate=5e-5,
    warmup_steps=0.1,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="balanced_accuracy",
    dataloader_num_workers=4,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

In [44]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
# srun --job-name=phastyle-sequence-db-embedding-v1 \
#      --nodes=1 \
#      --nodelist=lolka \
#      --ntasks-per-node=1 \
#      --cpus-per-task=8 \
#      --gres=gpu:2 \
#      --mem=150G \
#      --time=7-00:00:00 \
#      --partition=hi \
#      --output=/home/osvathm/prokbert/logs/%x_%j.out \
#      --error=/home/osvathm/prokbert/logs/%x_%j.err \
#      torchrun --nproc_per_node=2 examples/phastyle_training.py

# notes
# requested gpu count should match nproc_per_node param